# Vehicle Sales Data Cleaning

This notebook applies cleaning decisions based on findings from the initial data audit.

The raw dataset will remain unchanged. Cleaning will be performed on a separate copy of the data.

In [1]:
import pandas as pd

In [2]:
cars = pd.read_csv("../data/raw/car_prices.csv")

In [3]:
clean = cars.copy()

In [4]:
cars.shape

(558837, 16)

In [5]:
clean.shape

(558837, 16)

In [6]:
malformed_rows = (
    clean["transmission"].isin(["Sedan", "sedan"])
    & clean["color"].astype(str).str.fullmatch(r"\d+")
)

In [7]:
malformed_rows.sum()

np.int64(26)

In [8]:
removed_malformed = clean[malformed_rows].copy()

In [9]:
clean = clean[~malformed_rows].copy()

In [10]:
clean.shape

(558811, 16)

## 1. Remove Malformed Records

The audit identified 26 Volkswagen Jetta records with values shifted across multiple columns.

Rather than attempting to reconstruct these records using assumptions, they are removed from the cleaned dataset. The original raw dataset remains unchanged, and the removed records are retained separately for traceability.

## 2. Normalize Vehicle Condition Scores

The raw `condition` column uses two formats:

- Whole-number grades such as `1`, `2`, `3`, `4`, and `5`
- Decimal grades stored without the decimal point, such as `21` for `2.1` and `49` for `4.9`

A new `condition_clean` column will be created so the original condition values remain available for comparison.

In [11]:
clean["condition_clean"] = clean["condition"].copy()

In [12]:
condition_needs_conversion = clean["condition_clean"] > 5

In [13]:
clean.loc[condition_needs_conversion, "condition_clean"] = (
    clean.loc[condition_needs_conversion, "condition_clean"] / 10
)

In [14]:
sorted(clean["condition_clean"].dropna().unique())

[np.float64(1.0),
 np.float64(1.1),
 np.float64(1.2),
 np.float64(1.3),
 np.float64(1.4),
 np.float64(1.5),
 np.float64(1.6),
 np.float64(1.7),
 np.float64(1.8),
 np.float64(1.9),
 np.float64(2.0),
 np.float64(2.1),
 np.float64(2.2),
 np.float64(2.3),
 np.float64(2.4),
 np.float64(2.5),
 np.float64(2.6),
 np.float64(2.7),
 np.float64(2.8),
 np.float64(2.9),
 np.float64(3.0),
 np.float64(3.1),
 np.float64(3.2),
 np.float64(3.3),
 np.float64(3.4),
 np.float64(3.5),
 np.float64(3.6),
 np.float64(3.7),
 np.float64(3.8),
 np.float64(3.9),
 np.float64(4.0),
 np.float64(4.1),
 np.float64(4.2),
 np.float64(4.3),
 np.float64(4.4),
 np.float64(4.5),
 np.float64(4.6),
 np.float64(4.7),
 np.float64(4.8),
 np.float64(4.9),
 np.float64(5.0)]

In [15]:
clean[["condition_clean","condition"]].head(20)

,condition_clean,condition
0,5.0,5.0
1,5.0,5.0
2,4.5,45.0
3,4.1,41.0
4,4.3,43.0
5,1.0,1.0
6,3.4,34.0
7,2.0,2.0
8,4.2,42.0
9,3.0,3.0


In [16]:
make_mapping = (
    clean.dropna(subset=["make"])
    .assign(make_key=lambda df: df["make"].str.lower())
    .groupby("make_key")["make"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [17]:
clean["make_clean"] = clean["make"].str.lower().map(make_mapping)

In [18]:
clean["make_clean"].value_counts().head(40)

make_clean
Ford             93997
Chevrolet        60587
Nissan           54017
Toyota           39966
Dodge            30955
Honda            27351
Hyundai          21836
BMW              20793
Kia              18084
Chrysler         17485
Mercedes-Benz    17141
Jeep             15483
Infiniti         15305
Volkswagen       12579
Lexus            11980
GMC              10638
Mazda             8508
Cadillac          7629
Acura             5926
Audi              5877
Lincoln           5786
Buick             5121
Subaru            5103
Ram               4574
Pontiac           4524
Mitsubishi        4257
Volvo             3788
MINI              3224
Saturn            2841
Mercury           2023
Land Rover        1864
Scion             1687
Jaguar            1420
Porsche           1402
Suzuki            1078
FIAT               865
HUMMER             805
Saab               484
smart              396
Oldsmobile         384
Name: count, dtype: int64

In [19]:
clean["make_clean"].nunique(dropna=True)

66

In [20]:
body_mapping = (
    clean.dropna(subset=["body"])
    .assign(body_key=lambda df: df["body"].str.lower())
    .groupby("body_key")["body"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [21]:
clean["body_clean"] = clean["body"].str.lower().map(body_mapping)

In [22]:
clean["body"].value_counts().head(40)

body
Sedan            199437
SUV              119292
sedan             41906
suv               24552
Hatchback         21380
Minivan           21363
Coupe             14602
Wagon             13630
Crew Cab          13280
Convertible        8652
SuperCrew          7423
G Sedan            5999
hatchback          4857
SuperCab           4449
minivan            4166
Regular Cab        4067
Van                3958
Extended Cab       3824
Quad Cab           3436
coupe              3150
crew cab           3114
wagon              2499
convertible        1824
supercrew          1610
E-Series Van       1455
g sedan            1418
G Coupe            1263
Double Cab         1251
supercab            862
regular cab         783
extended cab        683
quad cab            659
van                 570
CrewMax Cab         445
King Cab            436
e-series van        368
double cab          350
g coupe             330
G Convertible       249
Genesis Coupe       232
Name: count, dtype: int64

In [23]:
clean["body_clean"].nunique(dropna=True)

45

In [24]:
clean.loc[
    clean["make"].notna() & (clean["make"] != clean["make_clean"]),
    ["make", "make_clean"]
].drop_duplicates().head(30)

,make,make_clean
468,lincoln,Lincoln
743,bmw,BMW
977,ford,Ford
1563,mitsubishi,Mitsubishi
2525,chevrolet,Chevrolet
2615,chrysler,Chrysler
2719,pontiac,Pontiac
3346,cadillac,Cadillac
3470,land rover,Land Rover
3519,mazda,Mazda


In [25]:
clean.loc[
    clean["body"].notna() & (clean["body"] != clean["body_clean"]),
    ["body", "body_clean"]
].drop_duplicates().head(30)

,body,body_clean
2596,sedan,Sedan
10491,convertible,Convertible
234616,minivan,Minivan
366047,suv,SUV
419190,van,Van
463408,g sedan,G Sedan
463411,g coupe,G Coupe
463449,hatchback,Hatchback
463454,king cab,King Cab
463603,supercrew,SuperCrew


In [26]:
clean["transmission"].value_counts(dropna=False)

transmission
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

In [27]:
clean["transmission_clean"] = clean["transmission"].copy()

In [28]:
clean["transmission_clean"].value_counts(dropna=False)

transmission_clean
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

In [29]:
clean["transmission"].value_counts(dropna=False)

transmission
automatic    475915
NaN           65352
manual        17544
Name: count, dtype: int64

## 4. Transmission Check

After removing the 26 malformed records, the `transmission` column contains only `automatic`, `manual`, and missing values.

No additional transmission cleaning is required.

In [30]:
clean["color"].value_counts(dropna=False).head(30)

color
black        110970
white        106673
silver        83389
gray          82857
blue          51139
red           43569
—             24685
green         11382
gold          11342
beige          9222
burgundy       8972
brown          6717
orange         2078
purple         1561
off-white      1449
yellow         1285
NaN             749
charcoal        479
turquoise       236
pink             42
lime             15
Name: count, dtype: int64

In [31]:
clean[clean["color"] == "—"][
    ["year", "make", "model", "body", "transmission", "color", "interior", "seller"]
    ].head(20)

,year,make,model,body,transmission,color,interior,seller
134,2013,Infiniti,G Sedan,G Sedan,automatic,—,black,nissan infiniti lt
154,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt
185,2013,Infiniti,G Sedan,G Sedan,automatic,—,beige,nissan infiniti lt
282,2013,Infiniti,G Sedan,G Sedan,automatic,—,black,nissan infiniti lt
322,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt
341,2013,Hyundai,Sonata,Sedan,automatic,—,beige,hyundai motor finance
350,2013,Infiniti,G Sedan,G Sedan,automatic,—,beige,nissan infiniti lt
362,2013,Mercedes-Benz,C-Class,Coupe,automatic,—,black,mercedes-benz financial services
376,2013,Infiniti,M,Sedan,automatic,—,black,nissan north america inc.
443,2013,Infiniti,G Coupe,G Coupe,automatic,—,black,nissan infiniti lt


In [32]:
clean.loc[clean["color"] == "—", "make"].value_counts().head(15)

make
Ford             9980
Nissan           3296
Mercedes-Benz    1709
Volkswagen       1119
Chevrolet         981
Toyota            975
Infiniti          898
Lincoln           854
Dodge             621
BMW               547
Lexus             415
Hyundai           363
Honda             320
Chrysler          319
Jeep              292
Name: count, dtype: int64

In [33]:
clean.loc[clean["color"] == "—", "year"].value_counts().head(15)

year
2012    6360
2014    6299
2013    6156
2011    2356
2010     700
2008     492
2007     415
2015     402
2009     376
2006     321
2005     254
2004     192
2003     109
2002      77
2000      51
Name: count, dtype: int64

In [34]:
clean.loc[clean["color"] == "—", "seller"].value_counts().head(15)

seller
ford motor credit company llc            2631
avis budget group                        1728
nissan-infiniti lt                       1547
mercedes-benz financial services         1285
ford motor credit company llc pd         1029
r hollenshead auto sales inc              846
lease plan usa                            830
vw credit                                 768
nissan motor acceptance corporation       764
wheels exchange llc                       747
nissan infiniti lt                        724
ford motor credit company                 665
ars/avis budget group                     651
ge fleet services for itself/servicer     605
ford-lincoln dealer program               491
Name: count, dtype: int64

In [35]:
clean["color_clean"] = clean["color"].replace("—", pd.NA)

In [39]:
clean["color_clean"].value_counts(dropna=False)

color_clean
black        110970
white        106673
silver        83389
gray          82857
blue          51139
red           43569
NaN           25434
green         11382
gold          11342
beige          9222
burgundy       8972
brown          6717
orange         2078
purple         1561
off-white      1449
yellow         1285
charcoal        479
turquoise       236
pink             42
lime             15
Name: count, dtype: int64

## 5. Exterior Color Investigation

The `color` column contains a large number of records represented by an em dash (`—`).

Inspection of these rows shows that the surrounding vehicle information appears normally aligned, unlike the previously removed malformed Volkswagen Jetta records.

This suggests that `—` may represent an unrecorded or unavailable exterior color rather than a corrupted row.

No replacement has been made yet. The distribution of these records across makes, sellers, and model years will be checked before deciding how to handle them.

## Cleaning Progress

- Removed 26 malformed Volkswagen Jetta records with shifted column values.
- Created a normalized vehicle condition field.
- Standardized capitalization for vehicle makes.
- Standardized capitalization for body styles.
- Confirmed that transmission contains only automatic, manual, and missing values after removing malformed records.
- Began investigating the `—` values in exterior color.

In [37]:
clean.loc[clean["color"] == "—", "make"].value_counts().head(15)

make
Ford             9980
Nissan           3296
Mercedes-Benz    1709
Volkswagen       1119
Chevrolet         981
Toyota            975
Infiniti          898
Lincoln           854
Dodge             621
BMW               547
Lexus             415
Hyundai           363
Honda             320
Chrysler          319
Jeep              292
Name: count, dtype: int64

In [40]:
clean.loc[clean["color"] == "—", "seller"].value_counts().head(15)

seller
ford motor credit company llc            2631
avis budget group                        1728
nissan-infiniti lt                       1547
mercedes-benz financial services         1285
ford motor credit company llc pd         1029
r hollenshead auto sales inc              846
lease plan usa                            830
vw credit                                 768
nissan motor acceptance corporation       764
wheels exchange llc                       747
nissan infiniti lt                        724
ford motor credit company                 665
ars/avis budget group                     651
ge fleet services for itself/servicer     605
ford-lincoln dealer program               491
Name: count, dtype: int64

In [41]:
clean.loc[clean["color"] == "—", "year"].value_counts().sort_index()

year
1992       1
1993       3
1994       5
1995       8
1996       4
1997      15
1998      11
1999      32
2000      51
2001      46
2002      77
2003     109
2004     192
2005     254
2006     321
2007     415
2008     492
2009     376
2010     700
2011    2356
2012    6360
2013    6156
2014    6299
2015     402
Name: count, dtype: int64

In [42]:
(clean["color"] == "—").sum()

np.int64(24685)

### Cleaning Decision

The em dash (`—`) appears across many manufacturers, sellers, and model years, while the surrounding vehicle records remain properly structured.

Because `—` does not represent an actual exterior color, it will be treated as a missing value rather than as a separate color category. The original `color` column will remain unchanged for traceability.

In [49]:
clean["color_clean"] = clean["color"].replace("—", pd.NA)

In [50]:
clean["color_clean"].value_counts(dropna=False).head(25)

color_clean
black        110970
white        106673
silver        83389
gray          82857
blue          51139
red           43569
NaN           25434
green         11382
gold          11342
beige          9222
burgundy       8972
brown          6717
orange         2078
purple         1561
off-white      1449
yellow         1285
charcoal        479
turquoise       236
pink             42
lime             15
Name: count, dtype: int64

In [51]:
print("Original missing colors:", clean["color"].isna().sum())
print("Cleaned missing colors:", clean["color_clean"].isna().sum())

Original missing colors: 749
Cleaned missing colors: 25434


In [52]:
high_999999 = clean[clean["odometer"] == 999999]

In [53]:
high_999999["year"].value_counts().sort_index()

year
1993    1
1996    1
1997    1
1998    1
1999    2
2001    5
2002    2
2003    8
2004    3
2005    8
2006    7
2007    6
2008    4
2009    6
2010    2
2011    3
2012    2
2013    6
2014    4
Name: count, dtype: int64

In [54]:
high_999999["seller"].value_counts().head(50)

seller
santander consumer                                    25
800 loan mart                                          8
wells fargo dealer services                            4
credit acceptance corp/vrs/southfield                  3
gm financial                                           3
hyundai motor finance                                  2
regional acceptance corporation / greenville           2
chrysler capital                                       2
kia motors finance                                     2
mei finance                                            1
fiserv/global lending services                         1
automotive credit corp                                 1
355 toyota                                             1
caprock auto remarketing                               1
lexus of silver spring                                 1
enterprise vehicle exchange / tra / rental / tulsa     1
onemain remarketing services                           1
innovate loan servicing 

In [55]:
high_999999[
    ["year", "make", "model", "condition_clean", "odometer", "mmr", "sellingprice", "seller"]
].head(50)

,year,make,model,condition_clean,odometer,mmr,sellingprice,seller
275,2013,Hyundai,Elantra Coupe,1.0,999999.0,8025.0,2500.0,hyundai motor finance
4626,2003,Chevrolet,Silverado 1500,2.0,999999.0,1425.0,700.0,800 loan mart
13317,2009,Chevrolet,Cobalt,NaN,999999.0,3375.0,400.0,mei finance
13480,2009,Dodge,Charger,1.0,999999.0,3850.0,1700.0,santander consumer
13568,2009,Dodge,Charger,1.0,999999.0,4150.0,5500.0,santander consumer
20838,2006,Kia,Amanti,1.0,999999.0,900.0,800.0,fiserv/global lending services
38465,2012,Nissan,Altima,1.0,999999.0,7375.0,800.0,santander consumer
39178,2009,Dodge,Grand Caravan,1.0,999999.0,3325.0,1500.0,santander consumer
40121,2006,Ford,F-150,1.9,999999.0,7500.0,3600.0,wells fargo dealer services
55492,2005,Nissan,Quest,1.9,999999.0,2250.0,2600.0,santander consumer


## 6. Odometer Investigation

The audit identified 72 records with an odometer value of exactly 999,999 miles.

These records span many vehicle years, makes, and models. The value is also concentrated among several finance and remarketing sellers, with Santander Consumer accounting for the largest share.

Because the exact value 999,999 appears repeatedly while most other extremely high mileage values occur only once or twice, it is likely acting as a special or unreliable odometer value rather than a precise mileage reading.

The original odometer column will remain unchanged, while 999,999 will be treated as missing in a cleaned odometer field.

In [60]:
clean["odometer_clean"] = clean["odometer"].copy()

clean.loc[
    clean["odometer_clean"] == 999999,
    "odometer_clean"
] = pd.NA

In [61]:
print("Original 999999 values:", (clean["odometer"] == 999999).sum())
print("Cleaned 999999 values:", (clean["odometer_clean"] == 999999).sum())
print("Original missing odometer:", clean["odometer"].isna().sum())
print("Cleaned missing odometer:", clean["odometer_clean"].isna().sum())

Original 999999 values: 72
Cleaned 999999 values: 0
Original missing odometer: 94
Cleaned missing odometer: 166


### Selling Price Investigation 
The audit included several cases of $1 selling price with high value of mmr, this value may be a data issue or placeholder for a unknown value or unofficial selling

In [62]:
clean["sellingprice"].nsmallest(30)

7566        1.0
48453       1.0
293223      1.0
348252      1.0
5184      100.0
25588     100.0
30275     100.0
196184    100.0
205309    100.0
227773    100.0
241038    100.0
281158    100.0
353308    100.0
384512    100.0
406428    100.0
409841    100.0
421787    100.0
423599    100.0
443651    100.0
449497    100.0
466143    100.0
502080    100.0
530434    100.0
503632    125.0
5164      150.0
5247      150.0
29060     150.0
29358     150.0
30233     150.0
30268     150.0
Name: sellingprice, dtype: float64

In [63]:
clean[clean["sellingprice"] <= 100][
    ["year", "make", "model", "condition_clean",
     "odometer_clean", "mmr", "sellingprice", "seller"]
].sort_values("sellingprice")

,year,make,model,condition_clean,odometer_clean,mmr,sellingprice,seller
7566,2002,Ford,F-350 Super Duty,2.0,NaN,7850.0,1.0,laa leasing
48453,2003,Mercedes-Benz,E-Class,2.1,1.0,7325.0,1.0,peoples credit company inc
293223,2014,Ford,E-Series Van,4.1,31886.0,20800.0,1.0,u-haul
348252,1995,Isuzu,Rodeo,1.0,254132.0,375.0,1.0,automotive remarketing inc
30275,1993,NaN,NaN,1.0,39750.0,1775.0,100.0,phh vehicle management services
196184,2004,Pontiac,Montana,2.0,106495.0,1700.0,100.0,first hawaiian bank
25588,2003,Chevrolet,Malibu,1.0,186893.0,950.0,100.0,lester glenn buick
5184,1997,ford,explorer,1.0,195748.0,500.0,100.0,purple heart services inc
227773,2009,NaN,NaN,2.0,1.0,3825.0,100.0,palm beach county sheriff's office
205309,2002,Chrysler,Sebring,1.0,94937.0,1325.0,100.0,wells fargo dealer services


In [64]:
clean[clean["sellingprice"] <= 100]["sellingprice"].value_counts().sort_index()

sellingprice
1.0       4
100.0    19
Name: count, dtype: int64

In [65]:
clean[clean["sellingprice"] <= 100]["seller"].value_counts().head(20)

seller
hertz remarketing                                   3
first hawaiian bank                                 2
wells fargo dealer services                         2
automotive remarketing inc                          2
select remarketing group llc/midwest title loans    2
purple heart services inc                           1
laa leasing                                         1
lester glenn buick                                  1
phh vehicle management services                     1
peoples credit company inc                          1
palm beach county sheriff's office                  1
melody toyota                                       1
credit acceptance corp/vrs/southfield               1
u-haul                                              1
pr auto loans llc                                   1
enterprise car sales/orange                         1
dt credit corporation                               1
Name: count, dtype: int64

In [66]:
clean["sellingprice_clean"] = clean["sellingprice"].copy()

In [69]:
clean.loc[
    clean["sellingprice_clean"] == 1,
    "sellingprice_clean"
] = pd.NA

In [70]:
clean.isna().sum().sort_values(ascending=False)

transmission          65352
transmission_clean    65352
color_clean           25434
body                  13195
body_clean            13195
condition             11794
condition_clean       11794
trim                  10651
model                 10399
make                  10301
make_clean            10301
interior                749
color                   749
odometer_clean          166
odometer                 94
sellingprice_clean       16
saledate                 12
mmr                      12
sellingprice             12
vin                       0
year                      0
seller                    0
state                     0
dtype: int64

In [73]:
missing_summary = pd.DataFrame({
    "missing_count": clean.isna().sum(),
    "missing_percent": clean.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
transmission,65352,11.694831
transmission_clean,65352,11.694831
color_clean,25434,4.551449
body,13195,2.361263
body_clean,13195,2.361263
condition,11794,2.110553
condition_clean,11794,2.110553
trim,10651,1.906011
model,10399,1.860915
make,10301,1.843378


In [74]:
important_columns = [
    "year",
    "make_clean",
    "model",
    "body_clean",
    "transmission",
    "condition_clean",
    "odometer_clean",
    "mmr",
    "sellingprice_clean"
]

clean[important_columns].isna().sum()

year                      0
make_clean            10301
model                 10399
body_clean            13195
transmission          65352
condition_clean       11794
odometer_clean          166
mmr                      12
sellingprice_clean       16
dtype: int64

In [75]:
missing_make = clean["make_clean"].isna()
missing_model = clean["model"].isna()
missing_body = clean["body_clean"].isna()

print("Missing make:", missing_make.sum())
print("Missing model:", missing_model.sum())
print("Missing body:", missing_body.sum())

print(
    "Missing all three:",
    (missing_make & missing_model & missing_body).sum()
)

Missing make: 10301
Missing model: 10399
Missing body: 13195
Missing all three: 10301


In [76]:
clean[
    missing_make | missing_model | missing_body
][
    ["year", "make_clean", "model", "body_clean",
     "transmission", "condition_clean", "odometer_clean",
     "mmr", "sellingprice_clean", "seller"]
].head(25)

,year,make_clean,model,body_clean,transmission,condition_clean,odometer_clean,mmr,sellingprice_clean,seller
72,2014,BMW,NaN,Sedan,automatic,4.3,12172.0,67000.0,66000.0,financial services remarketing (lease)
468,2013,Lincoln,mkt,NaN,automatic,4.1,74874.0,19300.0,17750.0,remarketing by ge/manheim southern california
742,2012,NaN,NaN,NaN,automatic,3.8,23208.0,47200.0,46000.0,financial services remarketing (lease)
743,2012,BMW,750i,NaN,automatic,4.0,50790.0,33900.0,33500.0,financial services remarketing (lease)
747,2012,NaN,NaN,NaN,automatic,3.3,19785.0,49500.0,46000.0,financial services remarketing (lease)
766,2012,NaN,NaN,NaN,automatic,3.7,48424.0,42300.0,43000.0,financial services remarketing (lease)
770,2012,BMW,750li,NaN,automatic,3.7,31762.0,45000.0,45000.0,financial services remarketing (lease)
793,2012,BMW,750i,NaN,automatic,4.9,53016.0,31300.0,34250.0,financial services remarketing (lease)
794,2012,BMW,750li,NaN,automatic,3.4,24739.0,47000.0,46750.0,financial services remarketing (lease)
798,2012,NaN,NaN,NaN,automatic,4.8,39825.0,58100.0,58500.0,financial services remarketing (lease)


In [77]:
missing_identity = (
    clean["make_clean"].isna()
    & clean["model"].isna()
    & clean["body_clean"].isna()
)

clean.loc[missing_identity, "seller"].value_counts().head(20)

seller
dtg operations inc                        296
financial services remarketing (lease)    271
credit acceptance corp/vrs/southfield     264
santander consumer                        245
wells fargo dealer services               164
fca canada inc                            163
dt credit corporation                     163
gm remarketing                            163
chrysler canada inc                       155
tdaf remarketing                          119
vpsi  inc                                 110
r hollenshead auto sales inc              103
ge fleet services for itself/servicer      95
dt inventory                               81
lease plan usa                             68
purple heart                               66
palm beach county sheriff's office         66
purple heart services inc                  62
automotive remarketing inc                 56
aaero sweet company                        56
Name: count, dtype: int64

In [78]:
clean.loc[missing_identity, "year"].value_counts().sort_index()

year
1982       2
1983       1
1984       4
1985       8
1986       9
1987       7
1988      11
1989      14
1990       4
1991       2
1992       9
1993      28
1994      34
1995      57
1996      89
1997     192
1998     229
1999     342
2000     681
2001      40
2002     111
2003     485
2004     826
2005    1114
2006    1492
2007    1350
2008     447
2009     183
2010     499
2011     332
2012     935
2013      54
2014     496
2015     214
Name: count, dtype: int64

In [79]:
missing_identity.sum() / len(clean) * 100

np.float64(1.8433781725842906)

In [80]:
clean["identity_missing"] = (
    clean["make_clean"].isna()
    & clean["model"].isna()
    & clean["body_clean"].isna()
)

In [81]:
clean["identity_missing"].value_counts()

identity_missing
False    548510
True      10301
Name: count, dtype: int64

In [82]:
clean["saledate"].head(10)

0    Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1    Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2    Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3    Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4    Thu Dec 18 2014 12:30:00 GMT-0800 (PST)
5    Tue Dec 30 2014 12:00:00 GMT-0800 (PST)
6    Wed Dec 17 2014 12:30:00 GMT-0800 (PST)
7    Tue Dec 16 2014 13:00:00 GMT-0800 (PST)
8    Thu Dec 18 2014 12:00:00 GMT-0800 (PST)
9    Tue Jan 20 2015 04:00:00 GMT-0800 (PST)
Name: saledate, dtype: str

In [83]:
clean["saledate"].dtype

<StringDtype(storage='python', na_value=nan)>

In [84]:
sale_date_text = clean["saledate"].str.replace(
    r"\s+\([A-Z]+\)$",
    "",
    regex=True
)

In [85]:
clean["saledate_clean"] = pd.to_datetime(
    sale_date_text,
    format="%a %b %d %Y %H:%M:%S GMT%z",
    errors="coerce",
    utc=True
)

In [86]:
clean[["saledate", "saledate_clean"]].head(10)

,saledate,saledate_clean
0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),2014-12-16 20:30:00+00:00
1,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),2014-12-16 20:30:00+00:00
2,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),2015-01-15 12:30:00+00:00
3,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),2015-01-29 12:30:00+00:00
4,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),2014-12-18 20:30:00+00:00
5,Tue Dec 30 2014 12:00:00 GMT-0800 (PST),2014-12-30 20:00:00+00:00
6,Wed Dec 17 2014 12:30:00 GMT-0800 (PST),2014-12-17 20:30:00+00:00
7,Tue Dec 16 2014 13:00:00 GMT-0800 (PST),2014-12-16 21:00:00+00:00
8,Thu Dec 18 2014 12:00:00 GMT-0800 (PST),2014-12-18 20:00:00+00:00
9,Tue Jan 20 2015 04:00:00 GMT-0800 (PST),2015-01-20 12:00:00+00:00


In [87]:
clean["saledate_clean"].dtype

datetime64[us, UTC]

In [88]:
print("Original missing dates:", clean["saledate"].isna().sum())
print("Cleaned missing dates:", clean["saledate_clean"].isna().sum())

Original missing dates: 12
Cleaned missing dates: 12


In [89]:
clean["sale_year"] = clean["saledate_clean"].dt.year

In [90]:
clean["vehicle_age"] = clean["sale_year"] - clean["year"]

In [91]:
clean[["year", "sale_year", "vehicle_age"]].head(20)

,year,sale_year,vehicle_age
0,2015,2014.0,-1.0
1,2015,2014.0,-1.0
2,2014,2015.0,1.0
3,2015,2015.0,0.0
4,2014,2014.0,0.0
5,2015,2014.0,-1.0
6,2014,2014.0,0.0
7,2014,2014.0,0.0
8,2014,2014.0,0.0
9,2014,2015.0,1.0


In [92]:
clean["vehicle_age"].describe()

count    558799.000000
mean          4.865223
std           3.944025
min          -1.000000
25%           2.000000
50%           3.000000
75%           7.000000
max          33.000000
Name: vehicle_age, dtype: float64

### Vehicle Age Validation

Vehicle age ranges from -1 to 33 years.

Negative ages are limited to -1. These values are reasonable because manufacturers often release the next model year before the calendar year changes. For example, a 2015 model may be sold in late 2014.

No vehicle-age values were removed or modified.

## 11. Standardize Model Capitalization

The audit found duplicate vehicle model categories caused by inconsistent capitalization.

As with make and body style, the most frequently used original spelling for each lowercase model name will be used as the standardized value.

In [93]:
model_mapping = (
    clean.dropna(subset=["model"])
    .assign(model_key=lambda df: df["model"].str.lower())
    .groupby("model_key")["model"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [94]:
clean["model_clean"] = clean["model"].str.lower().map(model_mapping)

In [95]:
print("Original unique models:", clean["model"].nunique())
print("Cleaned unique models:", clean["model_clean"].nunique())

Original unique models: 973
Cleaned unique models: 851


In [96]:
clean.loc[
    clean["model"].notna()
    & (clean["model"] != clean["model_clean"]),
    ["model", "model_clean"]
].drop_duplicates().head(30)

,model,model_clean
468,mkt,MKT
1563,galant,Galant
2596,malibu,Malibu
2719,g6,G6
3022,impala,Impala
3310,hhr,HHR
3346,sts,STS
3395,corvette,Corvette
3483,lancer,Lancer
3519,cx-7,CX-7


In [97]:
clean["trim"].value_counts(dropna=False).head(30)

trim
Base        55817
SE          43648
LX          20757
Limited     18367
LT          16915
XLT         16741
LS          13427
LE          12903
GLS         12331
S           11883
2.5 S       11212
SXT         11196
Touring     11111
NaN         10651
SEL         10479
EX           9472
Sport        9076
2.5          6908
SLT          6013
SV           5700
EX-L         5382
LTZ          5285
XL           4204
G37x         4172
V6           3963
3.5 SV       3629
SL           3519
328i         3420
Lariat       3395
Titanium     3025
Name: count, dtype: int64

In [100]:
clean["state"].value_counts(dropna=False).head(51)

state
fl    82945
ca    73148
pa    53907
tx    45913
ga    34750
nj    27784
il    23486
nc    21845
oh    21575
tn    20895
mo    16013
mi    15511
nv    12685
va    12027
md    11158
wi     9851
mn     9429
az     8741
co     7775
wa     7416
ma     6729
ny     5699
in     4325
sc     4251
ne     4013
on     3442
pr     2725
la     2191
ms     1851
ut     1836
qc     1245
hi     1237
or     1155
ab      928
nm      171
ok       72
ns       61
al       26
Name: count, dtype: int64

In [101]:
clean["interior"].value_counts(dropna=False).head(30)

interior
black        244325
gray         178578
beige         59758
tan           44093
—             17076
brown          8640
red            1359
blue           1138
silver         1099
NaN             749
off-white       480
purple          339
gold            324
white           252
green           245
burgundy        191
orange          145
yellow           20
Name: count, dtype: int64

In [102]:
print("Trim raw:", clean["trim"].nunique(dropna=True))
print("Trim lowercase:", clean["trim"].str.lower().nunique(dropna=True))

print("Interior raw:", clean["interior"].nunique(dropna=True))
print("Interior lowercase:", clean["interior"].str.lower().nunique(dropna=True))

print("State raw:", clean["state"].nunique(dropna=True))
print("State lowercase:", clean["state"].str.lower().nunique(dropna=True))

Trim raw: 1962
Trim lowercase: 1887
Interior raw: 17
Interior lowercase: 17
State raw: 38
State lowercase: 38


In [103]:
clean["interior_clean"] = clean["interior"].replace("—", pd.NA)

In [104]:
trim_mapping = (
    clean.dropna(subset=["trim"])
    .assign(trim_key=lambda df: df["trim"].str.lower())
    .groupby("trim_key")["trim"]
    .agg(lambda x: x.value_counts().idxmax())
)

clean["trim_clean"] = clean["trim"].str.lower().map(trim_mapping)

In [105]:
clean.loc[
    clean["trim"].notna()
    & (clean["trim"] != clean["trim_clean"]),
    ["trim", "trim_clean"]
].drop_duplicates().head(30)

,trim,trim_clean
770,750li,750Li
2042,SPORT,Sport
2596,ls,LS
3310,lt,LT
3341,v8 se,V8 SE
3346,v8,V8
3483,es,ES
3598,touring,Touring
3926,hse,HSE
4312,WRX STi,WRX STI


Raw dataset shape: (558837, 16)
Cleaned dataset shape: (558811, 30)
Rows removed: 26


Duplicate rows: 0


In [108]:
print("Condition range:",
      clean["condition_clean"].min(),
      "to",
      clean["condition_clean"].max())

print("999999 odometers remaining:",
      (clean["odometer_clean"] == 999999).sum())

print("$1 selling prices remaining:",
      (clean["sellingprice_clean"] == 1).sum())

print("Exterior color placeholders remaining:",
      (clean["color_clean"] == "—").sum())

print("Interior color placeholders remaining:",
      (clean["interior_clean"] == "—").sum())

Condition range: 1.0 to 5.0
999999 odometers remaining: 0
$1 selling prices remaining: 0
Exterior color placeholders remaining: 0
Interior color placeholders remaining: 0


In [109]:
print("Unique cleaned makes:", clean["make_clean"].nunique())
print("Unique cleaned models:", clean["model_clean"].nunique())
print("Unique cleaned body styles:", clean["body_clean"].nunique())
print("Unique cleaned trims:", clean["trim_clean"].nunique())

Unique cleaned makes: 66
Unique cleaned models: 851
Unique cleaned body styles: 45
Unique cleaned trims: 1887


In [110]:
cleaned_columns = [
    "year",
    "make_clean",
    "model_clean",
    "trim_clean",
    "body_clean",
    "transmission",
    "condition_clean",
    "odometer_clean",
    "color_clean",
    "interior_clean",
    "state",
    "mmr",
    "sellingprice_clean",
    "saledate_clean",
    "vehicle_age"
]

clean[cleaned_columns].isna().sum().sort_values(ascending=False)

transmission          65352
color_clean           25434
interior_clean        17825
body_clean            13195
condition_clean       11794
trim_clean            10651
model_clean           10399
make_clean            10301
odometer_clean          166
sellingprice_clean       16
saledate_clean           12
vehicle_age              12
mmr                      12
year                      0
state                     0
dtype: int64

## Create Processed Dataset

A final analysis-ready dataset will be created using the cleaned versions of each field.

The original raw dataset remains unchanged in `data/raw/`, so the processed dataset can use simpler column names without losing traceability.

In [111]:
processed = clean[
    [
        "year",
        "make_clean",
        "model_clean",
        "trim_clean",
        "body_clean",
        "transmission",
        "vin",
        "state",
        "condition_clean",
        "odometer_clean",
        "color_clean",
        "interior_clean",
        "seller",
        "mmr",
        "sellingprice_clean",
        "saledate_clean",
        "sale_year",
        "vehicle_age",
        "identity_missing"
    ]
].copy()

In [112]:
processed = processed.rename(columns={
    "make_clean": "make",
    "model_clean": "model",
    "trim_clean": "trim",
    "body_clean": "body",
    "condition_clean": "condition",
    "odometer_clean": "odometer",
    "color_clean": "color",
    "interior_clean": "interior",
    "sellingprice_clean": "sellingprice",
    "saledate_clean": "saledate"
})

In [113]:
processed.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,sale_year,vehicle_age,identity_missing
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,2014-12-16 20:30:00+00:00,2014.0,-1.0,False
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,2014-12-16 20:30:00+00:00,2014.0,-1.0,False
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,4.5,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,2015-01-15 12:30:00+00:00,2015.0,1.0,False
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,4.1,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,2015-01-29 12:30:00+00:00,2015.0,0.0,False
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,4.3,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,2014-12-18 20:30:00+00:00,2014.0,0.0,False


In [115]:
processed.shape

(558811, 19)

In [116]:
processed.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate', 'sale_year', 'vehicle_age',
       'identity_missing'],
      dtype='str')

In [117]:
processed.to_csv(
    "../data/processed/vehicle_sales_clean.csv",
    index=False
)

In [118]:
test_processed = pd.read_csv(
    "../data/processed/vehicle_sales_clean.csv"
)

print(test_processed.shape)
test_processed.head()

(558811, 19)


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,sale_year,vehicle_age,identity_missing
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,2014-12-16 20:30:00+00:00,2014.0,-1.0,False
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,2014-12-16 20:30:00+00:00,2014.0,-1.0,False
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,4.5,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,2015-01-15 12:30:00+00:00,2015.0,1.0,False
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,4.1,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,2015-01-29 12:30:00+00:00,2015.0,0.0,False
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,4.3,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,2014-12-18 20:30:00+00:00,2014.0,0.0,False
